In [ ]:
!pip install pandas nltk scikit-learn

In [ ]:
!wget -O spam.csv https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv

--2026-06-21 09:18:23--  https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [application/octet-stream]
Saving to: ‘spam.csv’

spam.csv            100%[===================>] 491.86K  --.-KB/s    in 0.007s  

2026-06-21 09:18:24 (72.6 MB/s) - ‘spam.csv’ saved [503663/503663]



In [ ]:
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')

# The raw file has extra junk columns — keep only the two we need
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print(df.shape)          # how many rows, columns
print(df.head())         # first 5 messages
print(df['label'].value_counts())   # how many spam vs ham

(5572, 2)
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()                              # "FREE Prize!!" -> "free prize!!"
    text = re.sub(r'http\S+|www\S+', '', text)        # remove links
    # Explicitly keep only lowercase English letters and whitespace characters, handling Unicode
    text = re.sub(r'[^a-z\s]', '', text, flags=re.UNICODE)
    text = text.strip()                               # remove extra spaces
    return text

df['clean_message'] = df['message'].apply(clean_text)
print(df[['message', 'clean_message']].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in  a wkly comp to win fa cup final...  
3        u dun say so early hor u c already then say  
4  nah i dont think he goes to usf he lives aroun...  


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in stop_words]
    return ' '.join(filtered)

df['clean_message'] = df['clean_message'].apply(remove_stopwords)
print(df['clean_message'].head())

0    go jurong point crazy available bugis n great ...
1                              ok lar joking wif u oni
2    free entry wkly comp win fa cup final tkts st ...
3                  u dun say early hor u c already say
4          nah dont think goes usf lives around though
Name: clean_message, dtype: object


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(lemmatized)

df['clean_message'] = df['clean_message'].apply(lemmatize_text)
print(df['clean_message'].head())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


0    go jurong point crazy available bugis n great ...
1                              ok lar joking wif u oni
2    free entry wkly comp win fa cup final tkts st ...
3                  u dun say early hor u c already say
4             nah dont think go usf life around though
Name: clean_message, dtype: object


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_num'] = le.fit_transform(df['label'])  # ham -> 0, spam -> 1

print(df[['label', 'label_num']].drop_duplicates())

  label  label_num
0   ham          0
2  spam          1


In [ ]:
print("Before:", df.shape)
df = df.drop_duplicates(subset='clean_message', keep='first')
print("After:", df.shape)

Before: (5572, 4)
After: (5066, 4)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)  # keep the 3000 most useful words
X = tfidf.fit_transform(df['clean_message']).toarray()
y = df['label_num']

print("Feature matrix shape:", X.shape)  # (number of messages, 3000)

Feature matrix shape: (5066, 3000)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (4052, 3000) Test: (1014, 3000)


In [ ]:
import joblib

# Save the cleaned dataframe
df.to_csv('sms_preprocessed.csv', index=False)

# Save the TF-IDF vectorizer itself (so future code can reuse it)
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Saved: sms_preprocessed.csv and tfidf_vectorizer.pkl")

Saved: sms_preprocessed.csv and tfidf_vectorizer.pkl


In [ ]:
import re

print("===== 1. SHAPE CHECK ====")
print("Total messages after cleaning:", df.shape[0])
assert df.shape[0] > 5000, "Dataset looks too small — recheck Part 5/10"

print("\n===== 2. TEXT CLEANING CHECK (eyeball this) ====")
print("Original :", df['message'].iloc[0])
print("Cleaned  :", df['clean_message'].iloc[0])

print("\n===== 3. NO LEFTOVER PUNCTUATION / NUMBERS ====")
sample = ' '.join(df['clean_message'].sample(50, random_state=1))
assert not re.search(r'[^\w\s]', sample), "Punctuation still present — recheck Part 6"
assert not re.search(r'\d', sample), "Numbers still present — recheck Part 6"
print("Clean — no punctuation or digits found in a 50-message sample.")

print("\n===== 4. STOPWORDS REMOVED ====")
common_stopwords = {'the', 'is', 'a', 'and', 'to', 'of', 'in'}
words_in_data = set(' '.join(df['clean_message']).split())
leftover = common_stopwords & words_in_data
print("Leftover common stopwords:", leftover if leftover else "None — good.")

print("\n===== 5. LABEL ENCODING CHECK ====")
print(df[['label', 'label_num']].drop_duplicates())
# Expect: ham -> 0, spam -> 1

print("\n===== 6. CLASS BALANCE (should be roughly 87% ham / 13% spam) ====")
print(df['label'].value_counts(normalize=True))

print("\n===== 7. DUPLICATES REMAINING ====")
print("Duplicate rows left:", df.duplicated(subset='clean_message').sum())

print("\n===== 8. TF-IDF MATRIX SHAPE ====")
print("X shape:", X.shape)
assert X.shape[0] == df.shape[0], "Row count mismatch between X and df!"
assert X.shape[1] == 3000, "Expected 3000 TF-IDF features — recheck Part 11"

print("\n===== 9. TRAIN/TEST SPLIT CHECK ====")
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))
# Both should be close to the overall ~87/13 split — that's what stratify=y guarantees

print("\nIf no AssertionError printed above, your pipeline (Steps 1-13) is solid.")

===== 1. SHAPE CHECK ====
Total messages after cleaning: 5066

===== 2. TEXT CLEANING CHECK (eyeball this) ====
Original : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned  : go jurong point crazy available bugis n great world la e buffet cine got amore wat

===== 3. NO LEFTOVER PUNCTUATION / NUMBERS ====
Clean — no punctuation or digits found in a 50-message sample.

===== 4. STOPWORDS REMOVED ====
Leftover common stopwords: None — good.

===== 5. LABEL ENCODING CHECK ====
  label  label_num
0   ham          0
2  spam          1

===== 6. CLASS BALANCE (should be roughly 87% ham / 13% spam) ====
label
ham     0.884524
spam    0.115476
Name: proportion, dtype: float64

===== 7. DUPLICATES REMAINING ====
Duplicate rows left: 0

===== 8. TF-IDF MATRIX SHAPE ====
X shape: (5066, 3000)

===== 9. TRAIN/TEST SPLIT CHECK ====
Train size: 4052  Test size: 1014
Train class balance:
 label_num
0    0.884501
1    0.115499
Name

In [ ]:
import re

print("===== 1. SHAPE CHECK ====")
print("Total messages after cleaning:", df.shape[0])
assert df.shape[0] > 5000, "Dataset looks too small — recheck Part 5/10"

print("\n===== 2. TEXT CLEANING CHECK (eyeball this) ====")
print("Original :", df['message'].iloc[0])
print("Cleaned  :", df['clean_message'].iloc[0])

print("\n===== 3. NO LEFTOVER PUNCTUATION / NUMBERS ====")
sample = ' '.join(df['clean_message'].sample(50, random_state=1))
assert not re.search(r'[^\w\s]', sample), "Punctuation still present — recheck Part 6"
assert not re.search(r'\d', sample), "Numbers still present — recheck Part 6"
print("Clean — no punctuation or digits found in a 50-message sample.")

print("\n===== 4. STOPWORDS REMOVED ====")
common_stopwords = {'the', 'is', 'a', 'and', 'to', 'of', 'in'}
words_in_data = set(' '.join(df['clean_message']).split())
leftover = common_stopwords & words_in_data
print("Leftover common stopwords:", leftover if leftover else "None — good.")

print("\n===== 5. LABEL ENCODING CHECK ====")
print(df[['label', 'label_num']].drop_duplicates())
# Expect: ham -> 0, spam -> 1

print("\n===== 6. CLASS BALANCE (should be roughly 87% ham / 13% spam) ====")
print(df['label'].value_counts(normalize=True))

print("\n===== 7. DUPLICATES REMAINING ====")
print("Duplicate rows left:", df.duplicated(subset='clean_message').sum())

print("\n===== 8. TF-IDF MATRIX SHAPE ====")
print("X shape:", X.shape)
assert X.shape[0] == df.shape[0], "Row count mismatch between X and df!"
assert X.shape[1] == 3000, "Expected 3000 TF-IDF features — recheck Part 11"

print("\n===== 9. TRAIN/TEST SPLIT CHECK ====")
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))
# Both should be close to the overall ~87/13 split — that's what stratify=y guarantees

print("\nIf no AssertionError printed above, your pipeline (Steps 1-13) is solid.")

===== 1. SHAPE CHECK ====
Total messages after cleaning: 5066

===== 2. TEXT CLEANING CHECK (eyeball this) ====
Original : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned  : go jurong point crazy available bugis n great world la e buffet cine got amore wat

===== 3. NO LEFTOVER PUNCTUATION / NUMBERS ====
Clean — no punctuation or digits found in a 50-message sample.

===== 4. STOPWORDS REMOVED ====
Leftover common stopwords: None — good.

===== 5. LABEL ENCODING CHECK ====
  label  label_num
0   ham          0
2  spam          1

===== 6. CLASS BALANCE (should be roughly 87% ham / 13% spam) ====
label
ham     0.884524
spam    0.115476
Name: proportion, dtype: float64

===== 7. DUPLICATES REMAINING ====
Duplicate rows left: 0

===== 8. TF-IDF MATRIX SHAPE ====
X shape: (5066, 3000)

===== 9. TRAIN/TEST SPLIT CHECK ====
Train size: 4052  Test size: 1014
Train class balance:
 label_num
0    0.884501
1    0.115499
Name

In [ ]:
print(df.shape)

(5066, 4)


In [ ]:
print("Total messages after cleaning:", df.shape[0])
assert df.shape[0] > 5000, "Dataset looks too small — recheck Part 5/10"

Total messages after cleaning: 5066


In [ ]:
print("Original :", df['message'].iloc[0])
print("Cleaned  :", df['clean_message'].iloc[0])

Original : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned  : go jurong point crazy available bugis n great world la e buffet cine got amore wat


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re
sample = ' '.join(df['clean_message'].sample(50, random_state=1))
assert not re.search(r'[^\w\s]', sample), "Punctuation still present — recheck Part 6"
assert not re.search(r'\d', sample), "Numbers still present — recheck Part 6"
print("Clean — no punctuation or digits found.")

Clean — no punctuation or digits found.


In [ ]:
common_stopwords = {'the', 'is', 'a', 'and', 'to', 'of', 'in'}
words_in_data = set(' '.join(df['clean_message']).split())
leftover = common_stopwords & words_in_data
print("Leftover common stopwords:", leftover if leftover else "None — good.")

Leftover common stopwords: None — good.


In [ ]:
print(df[['label', 'label_num']].drop_duplicates())

  label  label_num
0   ham          0
2  spam          1


In [ ]:
print(df['label'].value_counts(normalize=True))

label
ham     0.884524
spam    0.115476
Name: proportion, dtype: float64


In [ ]:
print("Duplicate rows left:", df.duplicated(subset='clean_message').sum())

Duplicate rows left: 0


In [ ]:
print("X shape:", X.shape)
assert X.shape[0] == df.shape[0], "Row count mismatch between X and df!"
assert X.shape[1] == 3000, "Expected 3000 TF-IDF features — recheck Part 11"

X shape: (5066, 3000)


In [ ]:
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train size: 4052  Test size: 1014
Train class balance:
 label_num
0    0.884501
1    0.115499
Name: proportion, dtype: float64
Test class balance:
 label_num
0    0.884615
1    0.115385
Name: proportion, dtype: float64
